In [ ]:
import numpy as np
l,TT,GG,TG= np.loadtxt('/workspace/Ubuntu/SGWBxCMB/class_public/output/cgwb_repeatli00_cl.dat',unpack=True)
f_NL = 100
Cl_xc = lambda i: TG[i-2]
Cl_AGWB = lambda i: GG[i-2]
Cl_CMB = lambda i: TT[i-2]

l_left, TT_left, GG_left, TG_left = np.loadtxt('/workspace/Ubuntu/SGWBxCMB/class_public/output/cgwb_repeatli_left00_cl.dat',unpack=True)
f_NL_left = 99
Cl_left_xc = lambda i: TG_left[i-2]
Cl_left_AGWB = lambda i: GG_left[i-2]
Cl_left_CMB = lambda i: TT_left[i-2]


l_right, TT_right, GG_right, TG_right = np.loadtxt('/workspace/Ubuntu/SGWBxCMB/class_public/output/cgwb_repeatli_right00_cl.dat',unpack=True)
f_NL_right = 101
Cl_right_xc = lambda i: TG_right[i-2]
Cl_right_AGWB = lambda i: GG_right[i-2]
Cl_right_CMB = lambda i: TT_right[i-2]


In [2]:
# suppose there is no noise
N_l_AGWB = lambda l: 0

# 第一步，计算 Cl 矩阵，即下面中间这个
$$F=\sum_{\ell}^{\ell_{\max }}\left(0, \frac{\partial C_{\ell}^{\mathrm{AGWB}}}{\partial f_{N L}}, \frac{\partial C_{\ell}^{\mathrm{XC}}}{\partial f_{N L}}\right) \mathbb{C}_{\ell}^{-1}\left(\begin{array}{c}
0 \\
\frac{\partial C_{\ell}^{\text {AGWB }}}{\partial f_{N L}} \\
\frac{\partial C_{\ell}^{\mathrm{X}}}{\partial f_{N L}}
\end{array}\right)
$$

In [ ]:
sigma_l_XC2 = lambda l: (Cl_xc(l) ** 2 + Cl_CMB(l) * (Cl_AGWB(l) + N_l_AGWB(l))) / (2 * l + 1)
sigma_l_AGWB_XC2 = lambda l: (2 / (2 * l + 1)) * ((Cl_AGWB(l) + N_l_AGWB(l)) * Cl_xc(l))
sigma_l_AGWB2 = lambda l: (2 / (2 * l + 1)) * ((Cl_AGWB(l) + N_l_AGWB(l)) ** 2)

def Cl_matrix(ell):
    sigma_AGWB2 = sigma_l_AGWB2(ell)
    sigma_AGWB_XC2 = sigma_l_AGWB_XC2(ell)
    sigma_XC2 = sigma_l_XC2(ell)
    
    return np.array([[sigma_AGWB2, sigma_AGWB_XC2],
                     [sigma_AGWB_XC2, sigma_XC2]])
这部分应该是错了的，因为不能简单的把三维变成二维，注意到 C_l 是要求逆矩阵的。

第二步，计算 Fisher Matrix 
$$F=\sum_{\ell}^{\ell_{\max }}\left(0, \frac{\partial C_{\ell}^{\mathrm{AGWB}}}{\partial f_{N L}}, \frac{\partial C_{\ell}^{\mathrm{XC}}}{\partial f_{N L}}\right) \mathbb{C}_{\ell}^{-1}\left(\begin{array}{c}
0 \\
\frac{\partial C_{\ell}^{\text {AGWB }}}{\partial f_{N L}} \\
\frac{\partial C_{\ell}^{\mathrm{X}}}{\partial f_{N L}}
\end{array}\right)
$$

In [ ]:
def derivative_Cl_AGWB(ell, side='center'):
    if side == 'left':
        return (Cl_left_AGWB(ell) - Cl_AGWB(ell)) / (f_NL_left - f_NL)
    elif side == 'right':
        return (Cl_right_AGWB(ell) - Cl_AGWB(ell)) / (f_NL_right - f_NL)
    elif side == 'center':
        return (Cl_right_AGWB(ell) - Cl_left_AGWB(ell)) / (f_NL_right - f_NL_left)
    else:
        raise ValueError("Invalid side. Choose 'left' or 'right'.")

def derivative_Cl_XC(ell, side='center'):
    if side == 'left':
        return (Cl_left_xc(ell) - Cl_xc(ell)) / (f_NL_left - f_NL)
    elif side == 'right':
        return (Cl_right_xc(ell) - Cl_xc(ell)) / (f_NL_right - f_NL)
    elif side == 'center':
        return (Cl_right_xc(ell) - Cl_left_xc(ell)) / (f_NL_right - f_NL_left)
    else:
        raise ValueError("Invalid side. Choose 'left' or 'right'.")

def fisher_matrix(ell_max):
    F = 0
    for ell in range(2, ell_max + 1):
        dCl_AGWB_left = derivative_Cl_AGWB(ell, side='left')
        dCl_AGWB_right = derivative_Cl_AGWB(ell, side='right')
        dCl_XC_left = derivative_Cl_XC(ell, side='left')
        dCl_XC_right = derivative_Cl_XC(ell, side='right')
        
        dCl_AGWB = (dCl_AGWB_left + dCl_AGWB_right) / 2
        dCl_XC = (dCl_XC_left + dCl_XC_right) / 2
        
        Cl_inv = np.linalg.inv(Cl_matrix(ell))
        
        F += np.array([dCl_AGWB, dCl_XC]) @ Cl_inv @ np.array([[dCl_AGWB], [dCl_XC]])
    
    return F

ell_max = 20  # Example value, adjust as needed
F = fisher_matrix(ell_max)
print(F)
